# Edge, GPU monitoring, Netron, TFLite/LiteRT and TensorFlow.js

This notebook presents different runtimes for ML models on a GPU, edge devices, IoT hardware and in the browser.


In [ ]:
import tensorflow as tf
from pathlib import Path

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:5000], y_train[:5000], epochs=1, batch_size=128)

Path("models").mkdir(exist_ok=True)
model.save("models/mnist_small.keras")
print("Saved Keras model: models/mnist_small.keras")


## Netron

Netron makes it easy to inspect the model architecture.

Setup:

1. Open `https://netron.app` and drag the file `models/mnist_small.keras`.
2. Install locally:

```bash
pip install netron
netron models/mnist_small.keras
```


In [ ]:
# Conversion to TensorFlow Lite / LiteRT
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("models/mnist_small.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model size:", len(tflite_model) / 1024, "KB")


## TensorFlow.js

TensorFlow.js allows models to run in the browser or Node.js.

Export can be done with `tensorflowjs_converter`:

```bash
pip install tensorflowjs
tensorflowjs_converter --input_format=keras models/mnist_small.keras tfjs_model/
```

In the browser, the model can be loaded with `tf.loadLayersModel(...)`.


## nvidia-smi — GPU monitoring

Basic commands to show locally:

```bash
nvidia-smi
nvidia-smi --loop=1
nvidia-smi dmon
nvidia-smi pmon
```


What can be monitored with nvidia-smi?
- GPU utilization;
- memory usage;
- processes using the GPU;
- temperature and power draw;
- whether the model is actually using the GPU;
- whether the bottleneck is GPU, CPU, disk, dataloader or batch size.


In [ ]:
# Checking GPU availability from TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


## Summary

- TFLite/LiteRT is used for edge and on-device inference: phones, IoT, embedded systems and industrial devices.
- TensorFlow.js is useful when the model should run directly in the browser.
- Netron helps understand model structure before deployment.
- `nvidia-smi` is the simplest tool for quickly checking whether the model really uses the GPU and how many resources it consumes.
